# UMWEO AI — fine-tune a small mining assistant

Fine-tunes **Qwen2.5-1.5B-Instruct** (small, open, not gated) on the mining
knowledge dataset using QLoRA, so it fits Kaggle's free T4 GPU.

**Before running:**
1. Kaggle right sidebar → *Accelerator* → **GPU T4 x2** (or P100)
2. *Add Input* → upload `train.jsonl` as a dataset named **umweo-mining-data**
3. Run all cells. Training takes roughly 10–30 min depending on dataset size.

In [ ]:
%pip install -q -U transformers datasets peft trl bitsandbytes accelerate
%pip uninstall -q -y torchao

In [ ]:
import glob
from datasets import load_dataset

# Find train.jsonl wherever Kaggle mounted the attached dataset.
matches = glob.glob("/kaggle/input/**/train.jsonl", recursive=True)
assert matches, "train.jsonl not found - attach the dataset via Add Input"
print("Using:", matches[0])

dataset = load_dataset("json", data_files=matches[0], split="train")
print(f"{len(dataset)} training examples")
print(dataset[0]["messages"][1]["content"][:200])

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # one GPU is plenty; two doubles memory use

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16)
model.to("cuda")

In [ ]:
from peft import LoraConfig, get_peft_model
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

lora = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

def tokenize(example):
    text = tokenizer.apply_chat_template(example["messages"], tokenize=False)
    return tokenizer(text, truncation=True, max_length=1024)

tokenized = dataset.map(tokenize, remove_columns=dataset.column_names)
collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

args = TrainingArguments(
    output_dir="/kaggle/working/umweo-lora",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    logging_steps=5,
    save_strategy="no",
    fp16=True,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized,
    data_collator=collator,
)
trainer.train()

In [ ]:
trainer.save_model("/kaggle/working/umweo-lora-final")
tokenizer.save_pretrained("/kaggle/working/umweo-lora-final")
print("Saved. Download from the Output tab of this notebook.")

In [ ]:
# Quick test of the fine-tuned model
from transformers import pipeline

chat = [
    {"role": "system", "content": "You are UMWEO AI, a friendly mining assistant for artisanal and small-scale miners in Zambia."},
    {"role": "user", "content": "What should I know about hazardous waste when mining?"},
]
pipe = pipeline("text-generation", model=trainer.model, tokenizer=tokenizer)
out = pipe(chat, max_new_tokens=300, do_sample=False)
print(out[0]["generated_text"][-1]["content"])

## After training

- **Download** the `umweo-lora-final` folder from the notebook's *Output* tab.
- To run it **on a phone / offline**, merge the LoRA into the base model and
  convert to GGUF with llama.cpp, then use a mobile runtime (e.g. llama.cpp
  Android bindings). This is the phase-two offline assistant path.
- Better results come from better data: the auto-generated Q&A pairs are a
  starting point. Adding a few hundred real miner questions with good answers
  (reviewed by a mining expert) will improve quality far more than more epochs.